# ♻️ Обучение модели YOLOv8 для AI Smart Bin (Умный Мусорный Бак)

Этот блокнот обучает нейросеть **YOLOv8** распознавать типы мусора (**Пластик, Бумага, Органика** и др.) и автоматически экспортирует её в файл **`best.onnx`** для нашего WPF C# приложения.

### ⚡ Инструкция по запуску в Google Colab (всего 3 шага):
1. **Включите бесплатную видеокарту (GPU):**
   * В верхнем меню Colab выберите: `Среда выполнения` (Runtime) ➔ `Сменить тип среды выполнения` (Change runtime type).
   * Выберите **T4 GPU** и нажмите кнопку **Сохранить**.
2. **Запустите обучение:**
   * В меню выберите: `Среда выполнения` ➔ `Выполнить всё` (или нажимайте `Shift + Enter` на каждой ячейке).
3. **Получите файл:**
   * По окончании обучения (~10–12 минут) файл **`best.onnx`** автоматически скачается в папку «Загрузки» вашего браузера!
   * Поместите его в папку `SmartBinApp` на вашем компьютере.

In [ ]:
# 1. Проверяем подключение видеокарты NVIDIA T4
!nvidia-smi

In [ ]:
# 2. Установка библиотеки Ultralytics (YOLOv8), Roboflow и ONNX
!pip install -q ultralytics roboflow onnx

### 📥 3. Загрузка размеченного датасета отходов
Датасет содержит тысячи изображений пластиковых бутылок, бумаги, картона и органики.

In [ ]:
import os
import urllib.request
import zipfile

# Если у вас есть аккаунт на roboflow.com, можете указать свой ключ ниже,
# либо оставьте пустым для автоматической загрузки проверенного открытого датасета:
ROBOFLOW_API_KEY = ""

if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("yolov8-ofcbj").project("waste-detection-saah1")
    dataset = project.version(1).download("yolov8")
    data_yaml = f"{dataset.location}/data.yaml"
else:
    print("📦 Клонирование репозитория с готовым датасетом отходов...")
    !git clone --depth 1 https://github.com/gianlucasposito/YOLO-Waste-Detection.git /content/waste_repo
    data_yaml = "/content/waste_repo/dataset/data.yaml"

print("✅ Датасет готов к обучению! Конфигурационный файл:", data_yaml)

### 🚀 4. Запуск обучения модели YOLOv8 nano
Модель оптимизирована под работу в реальном времени на веб-камере со скоростью 30+ кадров/сек.

In [ ]:
from ultralytics import YOLO

# Инициализируем легкую базовую модель YOLOv8 Nano
model = YOLO('yolov8n.pt')

# Запуск тонкой настройки (Fine-tuning) на 30 эпох
results = model.train(
    data=data_yaml,
    epochs=30,
    imgsz=640,
    batch=16,
    patience=8,
    save=True,
    device=0
)

print("🎉 Обучение успешно завершено!")

### 📊 5. Визуализация результатов детекции

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image

# Отображаем результаты валидации на тестовых изображениях
pred_images = glob.glob('runs/detect/train/val_batch0_pred.jpg') or glob.glob('runs/detect/train*/val_batch0_pred.jpg')
if pred_images:
    img = Image.open(pred_images[0])
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Примеры детекции мусора обученной моделью', fontsize=14)
    plt.show()
else:
    print("Графики сохранены в папке runs/detect/train/")

### 📦 6. Экспорт в ONNX и автоматическое скачивание файла

In [ ]:
import glob
from google.colab import files
from ultralytics import YOLO

# Находим лучшие веса (best.pt)
best_weights = glob.glob('runs/detect/train/weights/best.pt') or glob.glob('runs/detect/train*/weights/best.pt')

if best_weights:
    print(f"Найдена лучшая модель: {best_weights[0]}")
    best_model = YOLO(best_weights[0])
    
    # Экспортируем в ONNX с размером кадра 640x640
    onnx_file = best_model.export(format='onnx', imgsz=640, dynamic=False, opset=12)
    print(f"✅ Модель успешно экспортирована в ONNX: {onnx_file}")
    
    # Скачивание файла на компьютер
    print("⬇️ Запуск скачивания best.onnx в браузер...")
    files.download(onnx_file)
else:
    print("❌ Файл best.pt не найден. Проверьте логи обучения выше.")

### 🎯 7. Что делать с файлом `best.onnx` дальше:
1. Скопируйте скачанный файл `best.onnx` в папку вашего проекта:  
   `c:\Users\nadia\Documents\projschool\proj_school\SmartBinApp\`  
   (или в `SmartBinApp\bin\Debug\net8.0-windows\`).
2. Запустите приложение **SmartBinApp** в Visual Studio.
3. Приложение автоматически обнаружит `best.onnx` и загорится зеленый индикатор **✓ best.onnx**!
4. Включите камеру и поставьте галочку **«⚡ Автоматическая сортировка»**.